# Лабораторная работа №3

## Оценивание параметров и доверительные интервалы

**Курс:** Основы статистики  
**Тема:** Статистический вывод I — от выборки к генеральной совокупности  
**Дедлайн:** см. Google-таблицу учета успеваемости со списком студентов  
**Максимальный балл:** 100 (80 основных + 20 бонусных)

---

### Инструкция по выполнению

1. **Выбор датасета:** Выберите датасет согласно инструкции ниже (см. Задание 0)
2. **Выполнение заданий:** Пишите код в ячейках под каждым заданием
3. **Именование переменных:** Строго соблюдайте указанные имена переменных - они используются для автопроверки
4. **Комментарии:** Добавляйте комментарии к коду для пояснения логики
5. **Визуализации:** Все графики должны иметь подписи осей и заголовки
6. **Интерпретации:** Текстовые объяснения должны быть содержательными и соответствовать минимальной длине
7. **Сохранение:** После выполнения сохраните ноутбук и убедитесь, что все ячейки выполнены

---

### Структура оценивания

| Часть | Задание | Баллы |
|-------|---------|-------|
| **Часть А (Базовая)** | | **80** |
| | Задание 1: Подготовка данных | 10 |
| | Задание 2: Точечное оценивание | 12 |
| | Задание 3: Параметрические ДИ | 15 |
| | Задание 4: Bootstrap ДИ | 18 |
| | Задание 5: Влияние размера выборки | 15 |
| | Задание 6: Сравнение методов | 10 |
| **Часть Б (Бонусная)** | | **+20** |
| | Задание 7★: Многомерный анализ | 10 |
| | Задание 8★: Стратифицированный анализ | 10 |
| **Максимум** | | **100** |

**Оценка:**
Количество баллов, набранных за лабораторную работу, делится на 10 при выставлении итогового рейтинга в журнал учета успеваемости.

---

## Импорт библиотек

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, t, levene
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Для воспроизводимости
np.random.seed(42)

---

# ЗАДАНИЕ 0: Выбор датасета

### Определение номера варианта

Перейдите по ссылке в чате курса на Google Таблицу со списком студентов. Найдите свое ФИО в списке и запомните соответствующий порядковый номер (поле № п/п) в первом столбце. Заполните его в ячейке ниже и выполните ячейку. Если вы не можете найти себя в списке, обратитесь к своему преподавателю.

In [2]:
# TODO: Укажите ваш номер из списка группы
Student_ID = 467335

Теперь выполните следующую ячейку. Она вычислит номер датасета и выведет информацию о нем.

In [3]:
datasets = [
    ('California Housing', 'https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html'),
    ('Diabetes', 'https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html'),
    ('MPG (Auto)', 'https://archive.ics.uci.edu/dataset/9/auto+mpg'),
    ('Wine Quality (White)', 'https://archive.ics.uci.edu/dataset/186/wine+quality'),
    ('Boston Housing', 'https://www.kaggle.com/c/boston-housing'),
    ('Concrete Strength', 'https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength'),
    ('Energy Efficiency', 'https://archive.ics.uci.edu/dataset/242/energy+efficiency'),
    ('Real Estate Valuation', 'https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set'),
    ('Forest Fires', 'https://archive.ics.uci.edu/dataset/162/forest+fires'),
    ('Airfoil Self-Noise', 'https://archive.ics.uci.edu/dataset/291/airfoil+self+noise')
]

dataset_id = None if Student_ID is None else (Student_ID - 1) % len(datasets)
if dataset_id is None:
    print("ОШИБКА! Не указан порядковый номер студента в списке группы.")
else:
    print(f"Ваш датасет: #{dataset_id + 1} — '{datasets[dataset_id][0]}'")
    print(f"Ссылка на описание: {datasets[dataset_id][1]}")
    print("\n📖 Инструкция по загрузке датасета будет предоставлена отдельно.")

Ваш датасет: #5 — 'Boston Housing'
Ссылка на описание: https://www.kaggle.com/c/boston-housing

📖 Инструкция по загрузке датасета будет предоставлена отдельно.


---

# ЧАСТЬ А: БАЗОВЫЕ ЗАДАНИЯ (80 баллов)

---

## ЗАДАНИЕ 1: Подготовка данных и exploratory data analysis (10 баллов)

**Цель:** Загрузить данные, проанализировать несколько переменных и выбрать наиболее подходящую для дальнейшего анализа.

**Задачи:**
1. Загрузите свой датасет в переменную `df`
2. Выберите **3 количественные переменные** для анализа (сохраните их названия в `variables_analyzed`)
3. Для каждой переменной:
   - Проверьте на пропуски и выбросы (метод IQR)
   - Постройте гистограмму и Q-Q plot
   - Проведите тест Шапиро-Уилка на нормальность
   - Вычислите коэффициенты асимметрии (skewness) и эксцесса (kurtosis)
4. На основе анализа **выберите одну переменную**, которая **наиболее близка к нормальному распределению**
5. Обоснуйте свой выбор (минимум 150 символов)

**Переменные для проверки:**
- `df` — загруженный датасет (pd.DataFrame)
- `variables_analyzed` — список из 3 названий переменных (list[str])
- `normality_test_results` — словарь с результатами анализа (dict)
  - Структура: `{variable_name: {'p_value': float, 'skewness': float, 'kurtosis': float}}`
- `selected_variable` — название выбранной переменной (str)
- `data_clean` — очищенные данные выбранной переменной (np.array)
- `selection_justification` — обоснование выбора (str, ≥150 символов)

**Подсказки:**
- Выбросы по IQR: значения < Q1 - 1.5×IQR или > Q3 + 1.5×IQR
- Q-Q plot: `stats.probplot(data, dist="norm", plot=plt)`
- Тест Шапиро-Уилка: `shapiro(data)` → возвращает (statistic, p_value)
- Skewness: `data.skew()`, Kurtosis: `data.kurtosis()`
- Близость к нормальности: высокий p-value (> 0.05), |skewness| < 0.5, |kurtosis| < 1

In [ ]:
# TODO: Загрузите датасет
df = None

print(f"Размерность датасета: {df.shape}")
df.head()

In [ ]:
# TODO: Выберите 3 количественные переменные для анализа
variables_analyzed = []  # Например: ['feature1', 'feature2', 'feature3']

print(f"Анализируемые переменные: {variables_analyzed}")

In [ ]:
# TODO: Проведите анализ для каждой переменной

# Инициализация словаря результатов
normality_test_results = {}

# Ваш код здесь
# Для каждой переменной:
# 1. Очистите от пропусков и выбросов
# 2. Постройте гистограмму и Q-Q plot
# 3. Проведите тест Шапиро-Уилка
# 4. Вычислите skewness и kurtosis
# 5. Сохраните результаты в normality_test_results

# Пример структуры результатов:
# normality_test_results = {
#     'feature1': {'p_value': 0.15, 'skewness': 0.3, 'kurtosis': -0.5},
#     'feature2': {'p_value': 0.001, 'skewness': 1.8, 'kurtosis': 3.2},
#     'feature3': {'p_value': 0.08, 'skewness': -0.1, 'kurtosis': 0.2}
# }

# Вывод результатов
for var, results in normality_test_results.items():
    print(f"\n{var}:")
    print(f"  Shapiro p-value: {results['p_value']:.4f}")
    print(f"  Skewness: {results['skewness']:.4f}")
    print(f"  Kurtosis: {results['kurtosis']:.4f}")

In [ ]:
# TODO: Выберите переменную с наилучшей нормальностью
selected_variable = None  # Название переменной

# TODO: Очистите данные выбранной переменной от пропусков и выбросов
data_clean = None  # np.array

# TODO: Обоснуйте выбор (минимум 150 символов)
selection_justification = """
Ваше обоснование здесь. Объясните:
- Почему выбрали именно эту переменную?
- Какие критерии нормальности были решающими?
- Как результаты тестов подтверждают ваш выбор?
"""

# Проверка
print(f"Выбранная переменная: {selected_variable}")
print(f"Размер очищенной выборки: {len(data_clean)}")
print(f"\nОбоснование ({len(selection_justification)} символов):")
print(selection_justification)

---

## ЗАДАНИЕ 2: Точечное оценивание с анализом устойчивости (12 баллов)

**Цель:** Исследовать, как выбросы влияют на различные точечные оценки параметров.

**Задачи:**
1. Для выбранной переменной получите **исходные данные** (до удаления выбросов)
2. Вычислите для **исходных данных**:
   - Среднее арифметическое
   - Медиану
   - 10% усеченное среднее (trimmed mean)
   - Стандартное отклонение
   - MAD (median absolute deviation)
   - Стандартную ошибку среднего (SEM)
3. Вычислите те же оценки для **очищенных данных** (data_clean)
4. Сравните результаты: вычислите относительное изменение для среднего и медианы
5. Сделайте вывод о робастности (устойчивости) разных мер

**Переменные для проверки:**
- `estimates_original` — оценки для исходных данных (dict)
  - Ключи: 'mean', 'median', 'trimmed_mean', 'std', 'mad', 'sem'
- `estimates_clean` — оценки для очищенных данных (dict, та же структура)
- `relative_change_mean` — относительное изменение среднего в % (float)
- `relative_change_median` — относительное изменение медианы в % (float)
- `robustness_analysis` — вывод о робастности (str, ≥100 символов)

**Подсказки:**
- Получите исходные данные до удаления выбросов: `data_original = df[selected_variable].dropna().values`
- Trimmed mean: `stats.trim_mean(data, proportiontocut=0.1)`
- MAD: `stats.median_abs_deviation(data)`
- SEM: `stats.sem(data)`
- Относительное изменение: `|(new - old) / old| × 100%`

In [ ]:
# TODO: Получите исходные данные (до удаления выбросов)
data_original = None  # np.array

print(f"Размер исходной выборки: {len(data_original)}")
print(f"Размер очищенной выборки: {len(data_clean)}")
print(f"Удалено наблюдений: {len(data_original) - len(data_clean)}")

In [ ]:
# TODO: Вычислите все оценки для исходных данных
estimates_original = {
    'mean': None,
    'median': None,
    'trimmed_mean': None,
    'std': None,
    'mad': None,
    'sem': None
}

# Ваш код здесь

print("Оценки для исходных данных:")
for key, value in estimates_original.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# TODO: Вычислите все оценки для очищенных данных
estimates_clean = {
    'mean': None,
    'median': None,
    'trimmed_mean': None,
    'std': None,
    'mad': None,
    'sem': None
}

# Ваш код здесь

print("Оценки для очищенных данных:")
for key, value in estimates_clean.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# TODO: Вычислите относительные изменения
relative_change_mean = None  # в процентах
relative_change_median = None  # в процентах

# Ваш код здесь

print(f"Относительное изменение среднего: {relative_change_mean:.2f}%")
print(f"Относительное изменение медианы: {relative_change_median:.2f}%")

In [ ]:
# TODO: Сделайте вывод о робастности (минимум 100 символов)
robustness_analysis = """
Ваш анализ здесь. Ответьте на вопросы:
- Какая мера центральной тенденции более устойчива к выбросам?
- Насколько сильно выбросы влияют на среднее vs медиану?
- Какая мера разброса более робастна: std или MAD?
"""

print(f"Анализ робастности ({len(robustness_analysis)} символов):")
print(robustness_analysis)

---

## ЗАДАНИЕ 3: Параметрические доверительные интервалы (15 баллов)

**Цель:** Построить доверительные интервалы методом t-распределения и оценить их практическую значимость.

**Задачи:**
1. Проверьте предположение о нормальности очищенных данных (тест Шапиро-Уилка, α=0.05)
2. Постройте 95% доверительный интервал для среднего методом t-распределения
3. Также постройте 90% и 99% доверительные интервалы
4. Визуализируйте результаты: график с тремя ДИ разной ширины
5. Вычислите **коэффициент вариации ДИ**: (upper - lower) / mean × 100%
6. Дайте **практическую интерпретацию** в контексте вашего датасета

**Переменные для проверки:**
- `normality_assumption_met` — результат теста нормальности (bool: True если p > 0.05)
- `ci_90_parametric` — 90% ДИ (tuple: (lower, upper))
- `ci_95_parametric` — 95% ДИ (tuple: (lower, upper))
- `ci_99_parametric` — 99% ДИ (tuple: (lower, upper))
- `coefficient_of_variation_ci` — относительная ширина 95% ДИ в % (float)
- `practical_interpretation` — интерпретация в контексте данных (str, ≥200 символов)
- `fig_ci_levels` — фигура с визуализацией трех ДИ (matplotlib Figure)

**Подсказки:**
- Тест нормальности: `shapiro(data_clean)` → (statistic, pvalue)
- Параметрический ДИ: `t.interval(confidence, df=n-1, loc=mean, scale=sem)`
- SEM: `stats.sem(data)`

In [ ]:
# TODO: Проверьте нормальность
normality_assumption_met = None  # bool

# Ваш код здесь

print(f"Предположение о нормальности выполнено: {normality_assumption_met}")

In [ ]:
# TODO: Постройте параметрические ДИ для трех уровней доверия
ci_90_parametric = None  # (lower, upper)
ci_95_parametric = None  # (lower, upper)
ci_99_parametric = None  # (lower, upper)

# Ваш код здесь

print("Параметрические доверительные интервалы:")
print(f"90% ДИ: [{ci_90_parametric[0]:.4f}, {ci_90_parametric[1]:.4f}]")
print(f"95% ДИ: [{ci_95_parametric[0]:.4f}, {ci_95_parametric[1]:.4f}]")
print(f"99% ДИ: [{ci_99_parametric[0]:.4f}, {ci_99_parametric[1]:.4f}]")

In [ ]:
# TODO: Вычислите коэффициент вариации для 95% ДИ
coefficient_of_variation_ci = None  # в процентах

# Ваш код здесь

print(f"Коэффициент вариации 95% ДИ: {coefficient_of_variation_ci:.2f}%")

In [ ]:
# TODO: Визуализируйте три ДИ на одном графике
fig_ci_levels = None

# Ваш код здесь
# Постройте график, показывающий среднее и три ДИ разной ширины
# Используйте разные цвета для разных уровней доверия

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Дайте практическую интерпретацию (минимум 200 символов)
practical_interpretation = """
Ваша интерпретация здесь. Объясните:
- Что означает 95% ДИ в контексте вашего датасета?
- Как можно использовать этот интервал для принятия решений?
- Насколько точна оценка среднего (на основе коэффициента вариации)?
- Какой уровень доверия вы бы выбрали для практического применения и почему?
"""

print(f"Практическая интерпретация ({len(practical_interpretation)} символов):")
print(practical_interpretation)

---

## ЗАДАНИЕ 4: Bootstrap с исследованием сходимости (18 баллов)

**Цель:** Исследовать, как число bootstrap-итераций влияет на стабильность оценки доверительных интервалов.

**Задачи:**
1. Реализуйте bootstrap-процедуру для трех вариантов числа итераций: 1000, 5000, 10000
2. Для каждого варианта:
   - Создайте bootstrap-выборки (с возвращением)
   - Вычислите среднее для каждой выборки
   - Постройте 95% ДИ методом процентилей (2.5% и 97.5%)
   - Визуализируйте распределение bootstrap-средних
3. Исследуйте сходимость: постройте график зависимости ширины ДИ от числа итераций
4. Сделайте вывод о минимальном числе итераций для стабильного результата

**Переменные для проверки:**
- `bootstrap_means_1000` — массив bootstrap-средних для 1000 итераций (np.array)
- `bootstrap_means_5000` — массив bootstrap-средних для 5000 итераций (np.array)
- `bootstrap_means_10000` — массив bootstrap-средних для 10000 итераций (np.array)
- `ci_95_bootstrap_1000` — 95% ДИ для 1000 итераций (tuple)
- `ci_95_bootstrap_5000` — 95% ДИ для 5000 итераций (tuple)
- `ci_95_bootstrap_10000` — 95% ДИ для 10000 итераций (tuple)
- `ci_width_convergence` — список ширин ДИ (list[float])
- `convergence_analysis` — анализ сходимости (str, ≥150 символов)
- `fig_bootstrap_distributions` — фигура с распределениями (matplotlib Figure)
- `fig_convergence` — фигура с графиком сходимости (matplotlib Figure)

**Подсказки:**
- Bootstrap-выборка: `np.random.choice(data, size=len(data), replace=True)`
- Процентили: `np.percentile(bootstrap_means, [2.5, 97.5])`
- Ширина ДИ: `upper - lower`

In [ ]:
# TODO: Реализуйте bootstrap для трех вариантов числа итераций

np.random.seed(42)  # Для воспроизводимости

# Вариант 1: 1000 итераций
bootstrap_means_1000 = None  # np.array
# Ваш код здесь

# Вариант 2: 5000 итераций
bootstrap_means_5000 = None  # np.array
# Ваш код здесь

# Вариант 3: 10000 итераций
bootstrap_means_10000 = None  # np.array
# Ваш код здесь

print("Bootstrap-процедура завершена")
print(f"Средние значений bootstrap-средних:")
print(f"  1000 итераций: {np.mean(bootstrap_means_1000):.4f}")
print(f"  5000 итераций: {np.mean(bootstrap_means_5000):.4f}")
print(f"  10000 итераций: {np.mean(bootstrap_means_10000):.4f}")

In [ ]:
# TODO: Постройте 95% ДИ для каждого варианта
ci_95_bootstrap_1000 = None  # (lower, upper)
ci_95_bootstrap_5000 = None  # (lower, upper)
ci_95_bootstrap_10000 = None  # (lower, upper)

# Ваш код здесь

print("Bootstrap 95% доверительные интервалы:")
print(f"1000 итераций:  [{ci_95_bootstrap_1000[0]:.4f}, {ci_95_bootstrap_1000[1]:.4f}]")
print(f"5000 итераций:  [{ci_95_bootstrap_5000[0]:.4f}, {ci_95_bootstrap_5000[1]:.4f}]")
print(f"10000 итераций: [{ci_95_bootstrap_10000[0]:.4f}, {ci_95_bootstrap_10000[1]:.4f}]")

In [ ]:
# TODO: Вычислите ширины ДИ
ci_width_convergence = []  # [ширина для 1000, 5000, 10000]

# Ваш код здесь

print("Ширины доверительных интервалов:")
for n, width in zip([1000, 5000, 10000], ci_width_convergence):
    print(f"  {n} итераций: {width:.4f}")

In [ ]:
# TODO: Визуализируйте распределения bootstrap-средних
fig_bootstrap_distributions, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ваш код здесь
# Постройте гистограммы + KDE для трех вариантов
# Отметьте границы 95% ДИ на графиках

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Постройте график сходимости
fig_convergence = None

# Ваш код здесь
# Постройте график: ось X - число итераций, ось Y - ширина ДИ

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Проанализируйте сходимость (минимум 150 символов)
convergence_analysis = """
Ваш анализ здесь. Ответьте на вопросы:
- Как меняется ширина ДИ при увеличении числа итераций?
- При каком числе итераций результат стабилизируется?
- Какое минимальное число итераций вы рекомендуете для практики?
"""

print(f"Анализ сходимости ({len(convergence_analysis)} символов):")
print(convergence_analysis)

---

## ЗАДАНИЕ 5: Влияние размера выборки — теория vs практика (15 баллов)

**Цель:** Проверить теоретическое предсказание о том, как размер выборки влияет на ширину доверительного интервала.

**Теория:** Ширина ДИ пропорциональна 1/√n, то есть при увеличении размера выборки в 4 раза ширина ДИ должна уменьшиться в 2 раза.

**Задачи:**
1. Создайте подвыборки разных размеров: n = 30, 50, 100, 200, 500, 1000 (если позволяет датасет)
2. Для каждой подвыборки постройте 95% параметрический ДИ и вычислите его ширину
3. Вычислите **теоретическую ширину** для каждого n по формуле: width(n) = width(30) × √(30/n)
4. Постройте график: эмпирическая vs теоретическая зависимость
5. Вычислите корреляцию между теорией и практикой
6. Ответьте на вопрос: какой размер выборки нужен для 10% точности? (ширина ДИ ≤ 10% от среднего)

**Переменные для проверки:**
- `sample_sizes` — список размеров подвыборок (list[int])
- `ci_widths_empirical` — эмпирические ширины ДИ (list[float])
- `ci_widths_theoretical` — теоретические ширины ДИ (list[float])
- `theory_practice_correlation` — корреляция Пирсона (float)
- `required_sample_size_10percent` — требуемый размер выборки для 10% точности (int)
- `sample_size_recommendation` — практические рекомендации (str, ≥150 символов)
- `fig_sample_size_effect` — график зависимости (matplotlib Figure)

**Подсказки:**
- Случайная подвыборка: `np.random.choice(data_clean, size=n, replace=False)`
- Ширина ДИ: `upper - lower`
- Корреляция: `np.corrcoef(empirical, theoretical)[0, 1]`

In [ ]:
# TODO: Определите размеры подвыборок
sample_sizes = [30, 50, 100, 200, 500]  # Добавьте 1000, если позволяет датасет

# Проверка: достаточно ли данных?
if len(data_clean) < max(sample_sizes):
    print(f"Внимание: размер данных ({len(data_clean)}) меньше максимального размера подвыборки")
    sample_sizes = [s for s in sample_sizes if s <= len(data_clean)]

print(f"Размеры подвыборок: {sample_sizes}")

In [ ]:
# TODO: Для каждого размера постройте ДИ и вычислите ширину
np.random.seed(42)

ci_widths_empirical = []  # Эмпирические ширины

# Ваш код здесь
for n in sample_sizes:
    # 1. Создайте подвыборку размера n
    # 2. Постройте 95% параметрический ДИ
    # 3. Вычислите ширину и добавьте в список
    pass

print("Эмпирические ширины ДИ:")
for n, width in zip(sample_sizes, ci_widths_empirical):
    print(f"  n={n:4d}: {width:.4f}")

In [ ]:
# TODO: Вычислите теоретические ширины по формуле
ci_widths_theoretical = []  # Теоретические ширины

# Ваш код здесь
# Формула: width(n) = width(30) × √(30/n)

print("Теоретические ширины ДИ:")
for n, width in zip(sample_sizes, ci_widths_theoretical):
    print(f"  n={n:4d}: {width:.4f}")

In [ ]:
# TODO: Вычислите корреляцию между теорией и практикой
theory_practice_correlation = None  # float

# Ваш код здесь

print(f"Корреляция между теорией и практикой: {theory_practice_correlation:.4f}")

In [ ]:
# TODO: Постройте график: эмпирическая vs теоретическая зависимость
fig_sample_size_effect = None

# Ваш код здесь
# Постройте две линии: эмпирическую и теоретическую
# Добавьте легенду, подписи осей, заголовок

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Найдите требуемый размер выборки для 10% точности
required_sample_size_10percent = None  # int

# Ваш код здесь
# Найдите минимальное n, при котором ширина ДИ ≤ 0.1 × среднее

print(f"Требуемый размер выборки для 10% точности: {required_sample_size_10percent}")

In [ ]:
# TODO: Дайте практические рекомендации (минимум 150 символов)
sample_size_recommendation = """
Ваши рекомендации здесь. Обсудите:
- Насколько хорошо теория соответствует практике?
- Какой размер выборки рекомендуете для практических исследований?
- Как размер выборки влияет на стоимость и точность исследования?
"""

print(f"Рекомендации по размеру выборки ({len(sample_size_recommendation)} символов):")
print(sample_size_recommendation)

---

## ЗАДАНИЕ 6: Сравнение методов в сложных условиях (10 баллов)

**Цель:** Исследовать, как параметрические и непараметрические методы работают на данных, нарушающих предположение о нормальности.

**Задачи:**
1. Выберите другую переменную из вашего датасета, которая **явно НЕ нормальна**:
   - |skewness| > 1 или тест Шапиро-Уилка дает p < 0.01
2. Постройте для нее **три типа 95% ДИ**:
   - Параметрический ДИ для среднего (t-распределение)
   - Bootstrap ДИ для среднего
   - Bootstrap ДИ для медианы
3. Визуализируйте все три интервала на одном графике
4. Проведите критический анализ:
   - Какой метод дает самый узкий/широкий интервал?
   - Какому методу вы доверяете больше и почему?
5. Сформулируйте рекомендации по выбору метода

**Переменные для проверки:**
- `nonnormal_variable` — название переменной (str)
- `nonnormal_skewness` — коэффициент асимметрии (float)
- `ci_95_parametric_nonnormal` — параметрический ДИ для среднего (tuple)
- `ci_95_bootstrap_mean_nonnormal` — bootstrap ДИ для среднего (tuple)
- `ci_95_bootstrap_median_nonnormal` — bootstrap ДИ для медианы (tuple)
- `method_comparison_analysis` — критический анализ (str, ≥250 символов)
- `bootstrap_recommendation` — когда использовать bootstrap (str, ≥100 символов)
- `fig_method_comparison` — график сравнения методов (matplotlib Figure)

**Подсказки:**
- Bootstrap для медианы: вычисляйте `np.median()` вместо `np.mean()` для каждой выборки

In [ ]:
# TODO: Найдите переменную с явным нарушением нормальности
nonnormal_variable = None  # str

# Ваш код здесь
# Проверьте несколько переменных на skewness и результаты теста Шапиро-Уилка

# Загрузите и очистите данные этой переменной
data_nonnormal = None  # np.array

# Вычислите skewness
nonnormal_skewness = None  # float

print(f"Выбранная переменная: {nonnormal_variable}")
print(f"Коэффициент асимметрии: {nonnormal_skewness:.4f}")
print(f"Размер выборки: {len(data_nonnormal)}")

In [ ]:
# TODO: Постройте параметрический ДИ для среднего
ci_95_parametric_nonnormal = None  # (lower, upper)

# Ваш код здесь

print(f"Параметрический ДИ (среднее): [{ci_95_parametric_nonnormal[0]:.4f}, {ci_95_parametric_nonnormal[1]:.4f}]")
print(f"Ширина: {ci_95_parametric_nonnormal[1] - ci_95_parametric_nonnormal[0]:.4f}")

In [ ]:
# TODO: Постройте bootstrap ДИ для среднего
np.random.seed(42)

ci_95_bootstrap_mean_nonnormal = None  # (lower, upper)

# Ваш код здесь

print(f"Bootstrap ДИ (среднее): [{ci_95_bootstrap_mean_nonnormal[0]:.4f}, {ci_95_bootstrap_mean_nonnormal[1]:.4f}]")
print(f"Ширина: {ci_95_bootstrap_mean_nonnormal[1] - ci_95_bootstrap_mean_nonnormal[0]:.4f}")

In [ ]:
# TODO: Постройте bootstrap ДИ для медианы
np.random.seed(42)

ci_95_bootstrap_median_nonnormal = None  # (lower, upper)

# Ваш код здесь

print(f"Bootstrap ДИ (медиана): [{ci_95_bootstrap_median_nonnormal[0]:.4f}, {ci_95_bootstrap_median_nonnormal[1]:.4f}]")
print(f"Ширина: {ci_95_bootstrap_median_nonnormal[1] - ci_95_bootstrap_median_nonnormal[0]:.4f}")

In [ ]:
# TODO: Визуализируйте сравнение трех методов
fig_method_comparison = None

# Ваш код здесь
# Постройте график, показывающий три ДИ
# Используйте разные цвета и стили для разных методов
# Добавьте легенду и подписи

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Критический анализ (минимум 250 символов)
method_comparison_analysis = """
Ваш анализ здесь. Ответьте на вопросы:
- Какой метод дает самый узкий интервал? Самый широкий?
- Почему параметрический метод может быть неточен для этих данных?
- Чем отличаются ДИ для среднего и медианы?
- Какому методу вы доверяете больше и почему?
- Что произойдет, если использовать параметрический метод для сильно асимметричных данных?
"""

print(f"Анализ сравнения методов ({len(method_comparison_analysis)} символов):")
print(method_comparison_analysis)

In [ ]:
# TODO: Рекомендации по использованию bootstrap (минимум 100 символов)
bootstrap_recommendation = """
Ваши рекомендации здесь. Объясните:
- В каких ситуациях следует предпочесть bootstrap параметрическим методам?
- Когда параметрические методы все еще приемлемы?
"""

print(f"Рекомендации по использованию bootstrap ({len(bootstrap_recommendation)} символов):")
print(bootstrap_recommendation)

---

# ЧАСТЬ Б: БОНУСНЫЕ ЗАДАНИЯ (20 баллов)

Выполнение бонусных заданий необязательно, но позволяет получить дополнительные баллы.

---

## ⭐ ЗАДАНИЕ 7: Многомерный анализ — доверительные эллипсы (10 баллов)

**Цель:** Построить доверительные области для двумерного распределения.

**Задачи:**
1. Выберите две коррелированные количественные переменные (|r| > 0.5)
2. Постройте **доверительный эллипс** для двумерного нормального распределения (95%)
   - Вычислите ковариационную матрицу
   - Используйте собственные значения и собственные векторы для построения эллипса
3. Постройте **bootstrap доверительную область** методом convex hull:
   - Генерируйте 1000 bootstrap-выборок
   - Для каждой вычислите пару средних
   - Постройте выпуклую оболочку для 95% внутренних точек
4. Визуализируйте оба метода на одном scatter plot
5. Сравните площади областей

**Переменные для проверки:**
- `variable_pair` — пара переменных (tuple[str, str])
- `correlation` — корреляция Пирсона (float)
- `covariance_matrix` — ковариационная матрица 2×2 (np.array)
- `ellipse_area` — площадь эллипса (float)
- `bootstrap_hull_area` — площадь bootstrap-области (float)
- `area_ratio` — отношение площадей (float)
- `fig_confidence_ellipse` — визуализация (matplotlib Figure)

**Подсказки:**
- Ковариационная матрица: `np.cov(x, y)`
- Эллипс: `matplotlib.patches.Ellipse`
- Convex hull: `scipy.spatial.ConvexHull`

In [ ]:
# TODO: Выберите пару коррелированных переменных
variable_pair = None  # (var1, var2)

# Ваш код здесь

# Вычислите корреляцию
correlation = None

print(f"Выбранная пара: {variable_pair}")
print(f"Корреляция: {correlation:.4f}")

In [ ]:
# TODO: Вычислите ковариационную матрицу
covariance_matrix = None  # 2x2 np.array

# Ваш код здесь

print("Ковариационная матрица:")
print(covariance_matrix)

In [ ]:
# TODO: Постройте доверительный эллипс (параметрический метод)
ellipse_area = None

# Ваш код здесь

print(f"Площадь параметрического эллипса: {ellipse_area:.2f}")

In [ ]:
# TODO: Постройте bootstrap доверительную область
np.random.seed(42)

bootstrap_hull_area = None

# Ваш код здесь

print(f"Площадь bootstrap области: {bootstrap_hull_area:.2f}")

In [ ]:
# TODO: Сравните площади
area_ratio = None

# Ваш код здесь

print(f"Отношение площадей (bootstrap/parametric): {area_ratio:.2f}")

In [ ]:
# TODO: Визуализируйте оба метода
fig_confidence_ellipse = None

# Ваш код здесь
# Постройте scatter plot исходных данных
# Добавьте параметрический эллипс
# Добавьте bootstrap convex hull

plt.tight_layout()
plt.show()

---

## ⭐ ЗАДАНИЕ 8: Стратифицированный анализ по группам (10 баллов)

**Цель:** Исследовать гетерогенность данных и провести метаанализ оценок из разных групп.

**Задачи:**
1. Разделите данные на 3+ группы по категориальной переменной
2. Для каждой группы:
   - Вычислите среднее и 95% ДИ
   - Проверьте однородность дисперсий (тест Левена)
3. Визуализируйте: forest plot со средними и ДИ для всех групп
4. Проведите метаанализ:
   - Fixed-effects модель: взвешенное среднее по размерам групп
   - Вычислите общий 95% ДИ для объединенной оценки
5. Тест на гетерогенность:
   - Вычислите I² статистику (доля вариации между группами)
6. Интерпретируйте: можно ли объединять группы?

**Переменные для проверки:**
- `groups` — названия групп (list[str])
- `group_means` — средние по группам (dict: {group: mean})
- `group_cis` — ДИ по группам (dict: {group: (lower, upper)})
- `levene_test_pvalue` — p-value теста Левена (float)
- `pooled_estimate` — объединенная оценка (float)
- `pooled_ci_95` — ДИ объединенной оценки (tuple)
- `i_squared` — статистика гетерогенности 0-100% (float)
- `heterogeneity_interpretation` — можно ли объединять (str, ≥150 символов)
- `fig_forest_plot` — forest plot (matplotlib Figure)

**Подсказки:**
- Тест Левена: `levene(*[group1, group2, group3])`
- Fixed-effects: взвешенное среднее по размерам групп
- I²: мера гетерогенности, 0% = однородно, 100% = максимальная гетерогенность

In [ ]:
# TODO: Разделите данные на группы
groups = []  # Названия групп

# Ваш код здесь
# Выберите категориальную переменную
# Разделите данные по группам

print(f"Группы: {groups}")
print(f"Количество групп: {len(groups)}")

In [ ]:
# TODO: Вычислите средние и ДИ для каждой группы
group_means = {}  # {group_name: mean}
group_cis = {}    # {group_name: (lower, upper)}

# Ваш код здесь

print("Средние и 95% ДИ по группам:")
for group in groups:
    mean = group_means[group]
    ci = group_cis[group]
    print(f"  {group}: {mean:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")

In [ ]:
# TODO: Проверьте однородность дисперсий (тест Левена)
levene_test_pvalue = None

# Ваш код здесь

print(f"Тест Левена p-value: {levene_test_pvalue:.4f}")
if levene_test_pvalue > 0.05:
    print("Дисперсии однородны (p > 0.05)")
else:
    print("Дисперсии различаются (p ≤ 0.05)")

In [ ]:
# TODO: Проведите метаанализ (fixed-effects модель)
pooled_estimate = None  # Взвешенное среднее
pooled_ci_95 = None     # ДИ для объединенной оценки

# Ваш код здесь

print(f"Объединенная оценка: {pooled_estimate:.4f}")
print(f"95% ДИ: [{pooled_ci_95[0]:.4f}, {pooled_ci_95[1]:.4f}]")

In [ ]:
# TODO: Вычислите I² статистику
i_squared = None  # 0-100%

# Ваш код здесь

print(f"I² статистика: {i_squared:.2f}%")
if i_squared < 25:
    print("Низкая гетерогенность")
elif i_squared < 50:
    print("Умеренная гетерогенность")
elif i_squared < 75:
    print("Существенная гетерогенность")
else:
    print("Высокая гетерогенность")

In [ ]:
# TODO: Постройте forest plot
fig_forest_plot = None

# Ваш код здесь
# Постройте средние и ДИ для каждой группы
# Добавьте линию объединенной оценки

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Интерпретация гетерогенности (минимум 150 символов)
heterogeneity_interpretation = """
Ваша интерпретация здесь. Ответьте на вопросы:
- Можно ли объединять оценки из разных групп?
- Что показывает тест Левена и I² статистика?
- Какие выводы можно сделать о различиях между группами?
"""

print(f"Интерпретация гетерогенности ({len(heterogeneity_interpretation)} символов):")
print(heterogeneity_interpretation)

---

# Итоговая проверка

Убедитесь, что все следующие переменные определены и имеют правильный тип:

In [ ]:
# Проверка обязательных переменных (базовая часть)
required_vars_basic = [
    'Student_ID', 'df',
    # Задание 1
    'variables_analyzed', 'normality_test_results', 'selected_variable',
    'data_clean', 'selection_justification',
    # Задание 2
    'data_original', 'estimates_original', 'estimates_clean',
    'relative_change_mean', 'relative_change_median', 'robustness_analysis',
    # Задание 3
    'normality_assumption_met', 'ci_90_parametric', 'ci_95_parametric', 'ci_99_parametric',
    'coefficient_of_variation_ci', 'practical_interpretation', 'fig_ci_levels',
    # Задание 4
    'bootstrap_means_1000', 'bootstrap_means_5000', 'bootstrap_means_10000',
    'ci_95_bootstrap_1000', 'ci_95_bootstrap_5000', 'ci_95_bootstrap_10000',
    'ci_width_convergence', 'convergence_analysis',
    'fig_bootstrap_distributions', 'fig_convergence',
    # Задание 5
    'sample_sizes', 'ci_widths_empirical', 'ci_widths_theoretical',
    'theory_practice_correlation', 'required_sample_size_10percent',
    'sample_size_recommendation', 'fig_sample_size_effect',
    # Задание 6
    'nonnormal_variable', 'nonnormal_skewness',
    'ci_95_parametric_nonnormal', 'ci_95_bootstrap_mean_nonnormal',
    'ci_95_bootstrap_median_nonnormal',
    'method_comparison_analysis', 'bootstrap_recommendation', 'fig_method_comparison'
]

print("=" * 60)
print("ПРОВЕРКА БАЗОВЫХ ЗАДАНИЙ (Часть А)")
print("=" * 60)

missing_vars = []
for var in required_vars_basic:
    if var in globals() and globals()[var] is not None:
        print(f"✓ {var}")
    else:
        print(f"✗ {var} - НЕ ОПРЕДЕЛЕНА!")
        missing_vars.append(var)

if missing_vars:
    print(f"\n⚠️ ВНИМАНИЕ! Не определены {len(missing_vars)} переменных")
else:
    print("\n✅ Все базовые переменные определены!")

In [ ]:
# Проверка бонусных переменных (часть Б)
required_vars_bonus = [
    # Задание 7
    'variable_pair', 'correlation', 'covariance_matrix',
    'ellipse_area', 'bootstrap_hull_area', 'area_ratio',
    'fig_confidence_ellipse',
    # Задание 8
    'groups', 'group_means', 'group_cis',
    'levene_test_pvalue', 'pooled_estimate', 'pooled_ci_95',
    'i_squared', 'heterogeneity_interpretation', 'fig_forest_plot'
]

print("=" * 60)
print("ПРОВЕРКА БОНУСНЫХ ЗАДАНИЙ (Часть Б)")
print("=" * 60)

bonus_missing = []
for var in required_vars_bonus:
    if var in globals() and globals()[var] is not None:
        print(f"✓ {var}")
    else:
        print(f"✗ {var} - НЕ ОПРЕДЕЛЕНА")
        bonus_missing.append(var)

if bonus_missing:
    print(f"\n⚠️ Не определены {len(bonus_missing)} бонусных переменных")
    print("(Это нормально, если вы не делали бонусные задания)")
else:
    print("\n🌟 Все бонусные переменные определены!")

---

## Финальные инструкции

Перед отправкой работы убедитесь, что:

- ✅ Все ячейки выполнены (Kernel → Restart & Run All)
- ✅ Все обязательные переменные определены
- ✅ Визуализации имеют подписи осей, заголовки и легенды
- ✅ Текстовые интерпретации соответствуют минимальной длине
- ✅ Код прокомментирован и читаем
- ✅ Нет ошибок при выполнении
